In [ ]:
class ProbeConfig:
    def __init__(self):
        # Define your configuration parameters
        self.batch_size = 32
        self.input_dim = 64
        self.nepochs = 30
        self.supervision = "U"
        self.device = device
        self.probe_type = "linear"
        self.ntries = 5
        self.lr = 1e-3
        self.weight_decay = 0.01
        self.with_direction = True
        self.dropout = 0.0

probe_config = ProbeConfig()

In [ ]:
import random

# Define a list of true facts
true_facts = [
    ("Paris", "France", "capital"),
    ("Berlin", "Germany", "capital"),
    ("Tokyo", "Japan", "capital"),
    ("London", "England", "capital"),
    ("Canberra", "Australia", "capital"),
    ("Rome", "Italy", "capital")
] * 16

false_facts = [
    ("Paris", "Brazil", "capital"),
    ("Berlin", "Greece", "capital"),
    ("Tokyo", "Spain", "capital"),
    ("London", "Nigeria", "capital"),
    ("Canberra", "USA", "capital"),
    ("Rome", "China", "capital")
] * 16

# Function to generate a sentence and label
def generate_fact(fact, label, neg=False):
    city, country, relation = fact
    statement = f"{city} is the {relation} of {country}"
    if neg:
        statement = f"{city} is not the {relation} of {country}"
        label = 1-label
    return statement, label

def generate_false_fact(fact, label, neg=False):
    city, country, relation = fact
    statement = f"{city} is the {relation} of {country}"
    if neg:
        statement = f"{city} is not the {relation} of {country}"
        label = 1-label
    return statement

# Generate the correct dataset
correct_dataset = []
corrupt_dataset = []

for fact in true_facts:
    correct_dataset.append(generate_fact(fact, 1))
    correct_dataset.append(generate_fact(fact, 1, neg=True))
for fact in false_facts:
    corrupt_dataset.append(generate_false_fact(fact, 0))
    corrupt_dataset.append(generate_false_fact(fact, 0, neg=True))
# Generate the corrupt dataset by swapping the country

# Shuffle the datasets
random.shuffle(correct_dataset)
random.shuffle(corrupt_dataset)

# Separate the data into x and y for model input
x0, y = zip(*correct_dataset)
x1 = corrupt_dataset

In [ ]:
model.reset_hooks()
#                                Our model   List of sents Aligned Labels
correct_extractor = ActivationExtractor(model=model, data=x0, labels=y)
correct_extractor.set_hooks([0, 1, 2, 3, 4, 5], ['z'], attn=True) # for instance
corrupt_extractor = ActivationExtractor(model=model, data=x1, labels=y)
corrupt_extractor.set_hooks([0, 1, 2, 3, 4, 5], ['z'], attn=True) # for instance

correct_activations, labels = correct_extractor.process() # Get
corrupt_activations, _ = corrupt_extractor.process()
correct_heads = [decompose_mha(x) for x in correct_activations.values()]
corrupt_heads = [decompose_mha(x) for x in corrupt_activations.values()]

# correct/corrupt heads is a list (layers) of lists (heads) of tensors (n_batch d_batch d_head)
combined_heads = [
    [
        (correct_heads[layer][head], corrupt_heads[layer][head])
        for head in range(len(correct_heads[layer]))
    ]
    for layer in range(len(correct_heads))
]


In [ ]:
'''
TEST (only accuracies)

accuracies_per_layer = []

for layer in model.cfg.layers:

  accuracies, _, _ = probe_sweep(attn_heads, labels, probe_config)
  accuracies_per_layer.append(accuracies)

'''
tot_accuracies = []
tot_directions = []
tot_probes = []

for layer in range(6):
  accuracies, directions, probes = probe_sweep(combined_heads[layer], labels, probe_config)
  tot_accuracies.append(accuracies)
  tot_directions.append(directions)
  tot_probes.append(probes)

# We should get: probes trained on each set with (x_train = tensor of shape (n_batch d_batch head_size); y_train = tensor of shape (n_batch d_batch))

print(tot_accuracies)

In [ ]:
import seaborn as sns

print(tot_accuracies)

plt.figure(figsize=(8, 6))  # Optional: Adjust figure size
sns.heatmap(tot_accuracies, annot=True, fmt=".2f", cmap="coolwarm")  # Annotates values on the heatmap
plt.title("Matrix Heatmap")  # Add a title
plt.xlabel("Heads")  # X-axis label
plt.ylabel("Layers")  # Y-axis label
plt.show()